In [0]:
from pyspark.sql import SparkSession

In [0]:
spark=(
    SparkSession
    .builder
    .appName("Spark DataFrames")
    .getOrCreate()
)

In [0]:
emp_data = [
    ("001","101","John Doe",30,"Male",50000,"2015-01-01"),
    ("002","101","Jane Smith",25,"Female",45000,"2016-02-15"),
    ("003","102","Bob Brown",35,"Male",55000,"2014-05-01"),
    ("004","102","Alice Lee",28,"Female",48000,"2017-09-30"),
    ("005","103","Jack Chan",40,"Male",60000,"2013-04-01"),
    ("006","103","Jill Wong",32,"Female",52000,"2018-07-01"),
    ("007","101","James Johnson",42,"Male",70000,"2012-03-15"),
    ("008","102","Kate Kim",29,"Female",51000,"2019-10-01"),
    ("009","103","Tom Tan",33,"Male",58000,"2016-06-01"),
    ("010","104","Lisa Lee",27,"Female",47000,"2018-08-01"),
    ("011",None,"David Park",38,"Male",65000,"2015-11-01"),   # bad/missing dept_id — intentional
    ("012","105","Susan Chen",31,"Female",54000,"2017-02-15"),
]
emp_schema = "employee_id string, department_id string, name string, age int, gender string, salary int, hire_date string"

In [0]:
emp=spark.createDataFrame(data=emp_data,schema=emp_schema)
display(emp)

In [0]:
emp.printSchema()

In [0]:
# Casting Column
# select employee_id, name, age, cast(salary as double) as salary from emp

emp_casted=emp.selectExpr("employee_id", "name as Fullname", "age", "cast(salary as double) as salary")
emp_casted.show()
emp_casted.printSchema()


## selectExpr() in PySpark

### What is selectExpr()?
`selectExpr()` is a PySpark DataFrame method that allows you to select columns using SQL expressions as strings. It's a variant of the `select()` method that accepts SQL-like syntax.

### Why Use selectExpr()?
- **SQL-friendly syntax**: If you're comfortable with SQL, you can write transformations using familiar SQL expressions
- **Concise code**: Perform complex transformations in a single string expression
- **Multiple operations at once**: Combine column selection, casting, calculations, and aliasing in one call
- **Quick prototyping**: Faster to write SQL-style expressions without importing multiple PySpark functions

### How is selectExpr() Different from select()?

| Aspect | `select()` | `selectExpr()` |
|--------|-----------|----------------|
| **Syntax** | Uses Column objects and PySpark functions | Uses SQL expressions as strings |
| **Example** | `df.select(col("name"), (col("salary") * 1.1).alias("new_salary"))` | `df.selectExpr("name", "salary * 1.1 as new_salary")` |
| **Imports needed** | Often requires importing `col`, `lit`, etc. | No imports needed |
| **Type checking** | Compile-time checking (IDE support) | Runtime string evaluation |
| **Flexibility** | More programmatic, better for dynamic logic | Better for quick SQL-style transformations |

### How to Use selectExpr()?

**Basic column selection:**
```python
df.selectExpr("employee_id", "name", "salary")
```

**Casting columns:**
```python
df.selectExpr("employee_id", "cast(salary as double) as salary")
```

**Calculations and transformations:**
```python
df.selectExpr("employee_id", "name", "salary * 0.2 as tax", "salary * 1.1 as increased_salary")
```

**Aliasing columns:**
```python
df.selectExpr("employee_id", "name as full_name", "age")
```

**Complex expressions:**
```python
df.selectExpr(
    "employee_id",
    "upper(name) as name_upper",
    "year(current_date()) - year(hire_date) as years_employed"
)
```

### When to Choose Which?
- Use **`selectExpr()`** when you want quick, SQL-like transformations and are comfortable with SQL syntax
- Use **`select()`** when you need programmatic control, IDE autocomplete, or complex logic with Python variables


In [0]:
# Adding Columns
# select employee_id, name, age, salary, (salary * 0.2) as tax from emp_casted

emp_taxation_details=emp_casted.selectExpr("employee_id", "Fullname", "age", "salary", " (salary * 0.2) as tax", "salary-(salary*0.2) as SalaryAfterTax")
emp_taxation_details.show()


In [0]:
from pyspark.sql.functions import col
emp_taxation_details.select(col("employee_id"), "Fullname",'salary').where(col("salary").between(40000,50000) & col("Fullname").contains("a")).show()